# AI Model Experiment & Evaluation
**_Sentiment Analysis_ pada Ulasan Pelanggan _E-Commerce_**

*Notebook* ini berisi eksperimen dan evaluasi dua pendekatan AI untuk menyelesaikan task klasifikasi sentimen ulasan pelanggan:
1. Model klasik *Machine Learning* (Scikit-learn)
2. LLM API (Gemini)

Kedua pendekatan dijalankan pada *dataset* dan *test set* yang sama, kemudian dibandingkan performanya untuk memperoleh rekomendasi *technical approach*.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

**Objective**

Sebuah tim produk *e-commerce* ingin menghadirkan fitur otomatis untuk mengklasifikasikan sentimen ulasan pelanggan pada halaman produk, sehingga proses *monitoring* kepuasan pelanggan tidak lagi bergantung pada pembacaan ulasan secara manual satu per satu. Sebelum fitur ini dibangun ke sistem produksi, perlu dilakukan eksperimen awal untuk membandingkan dua kandidat pendekatan AI, yaitu model klasik *Machine Learning* yang dilatih sendiri menggunakan Scikit-learn dan *Large Language Model* (LLM) yang diakses melalui Gemini AI API menggunakan pendekatan *prompting*, sehingga dapat diperoleh rekomendasi pendekatan yang paling sesuai untuk dikembangkan lebih lanjut.

**Target/Label**

Target yang diprediksi adalah label sentimen dari sebuah ulasan pelanggan, berupa klasifikasi biner dengan dua kelas, yaitu `positif` dan `negatif`. Label ini bersumber dari kolom `sentiment` pada *dataset*, sedangkan fitur yang digunakan sebagai input berasal dari kolom `review_text`.

**Batasan dan Asumsi**

- *Dataset* yang digunakan adalah *dataset* ulasan pelanggan yang sudah dibagikan (`customer_reviews_sentiment.csv`) dan tidak dilakukan pencarian atau penggunaan *dataset* lain.
- *Dataset* diasumsikan sudah bersih dan setiap barisnya hanya memiliki satu label sentimen (positif atau negatif), tanpa kelas netral.
- Kedua pendekatan dievaluasi menggunakan *test set* yang sama agar perbandingan performa adil.
- Pendekatan model klasik hanya menggunakan teknik *Machine Learning* konvensional (bukan *Deep Learning*), sesuai dengan cakupan pembelajaran pada sesi ini.
- Pendekatan LLM API menggunakan pendekatan *prompting* (*few-shot*) tanpa proses *fine-tuning* maupun *training* tambahan.

## 2. Import Library & Load Dataset

In [1]:
# Memuat library yang dibutuhkan serta dataset ulasan pelanggan
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

# Memuat variabel dari file .env (menyimpan GEMINI_API_KEY)
load_dotenv()

df = pd.read_csv('../data/customer_reviews_sentiment.csv')
df.head()

,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

In [2]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

Train size: 160
Test size: 40


## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [3]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### 4.2 Training Model

In [4]:
clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

### 4.3 Prediksi pada Test Set

In [5]:
y_pred_classic = clf.predict(X_test_vec)

## 5. Pendekatan 2 — LLM API (Gemini)

### 5.1 Setup API

In [6]:
# Mengambil API Key dari environment variable
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

### 5.2 Merancang Prompt

*Prompt* dirancang menggunakan pendekatan **_few-shot prompting_**, yaitu memberikan beberapa contoh pasangan ulasan dan label sentimen sebelum meminta model mengklasifikasikan ulasan baru. Pendekatan ini dipilih dibandingkan *zero-shot* karena:

- Contoh *few-shot* membantu model memahami konteks spesifik domain ulasan *e-commerce* berbahasa Indonesia, termasuk gaya bahasa informal dan istilah-istilah yang umum dipakai pembeli (misal: "barang sesuai deskripsi", "pengiriman lambat").
- Contoh *few-shot* juga memperjelas format *output* yang diharapkan, sehingga mengurangi variasi jawaban model dan mempermudah proses *parsing* hasil prediksi.
- Dengan menunjukkan pola input-output secara eksplisit, konsistensi label yang dihasilkan model (`positif`/`negatif`) menjadi lebih terjaga dibandingkan *zero-shot* yang hanya mengandalkan instruksi tanpa contoh.

Struktur *prompt* yang digunakan terdiri atas empat bagian:
1. **Persona**: memposisikan model sebagai asisten analis sentimen yang ahli menilai ulasan pelanggan *e-commerce*.
2. **Task**: instruksi untuk mengklasifikasikan sentimen ulasan ke dalam dua kelas.
3. **Konteks**: beberapa contoh few-shot pasangan ulasan dan label sentimennya.
4. **Output**: instruksi tegas agar jawaban hanya berupa satu kata, yaitu `positif` atau `negatif`, tanpa tambahan penjelasan.

In [7]:
# Contoh few-shot yang merepresentasikan pola ulasan positif dan negatif
FEW_SHOT_EXAMPLES = [
    ("Pengiriman cepat, barang sesuai deskripsi, dan kualitasnya bagus.", "positif"),
    ("Barang datang dalam kondisi rusak dan seller sulit dihubungi.", "negatif"),
    ("Ukuran pas dan bahannya nyaman dipakai, recommended seller.", "positif"),
    ("Proses pengiriman lambat dan tidak sesuai estimasi yang dijanjikan.", "negatif"),
]

def build_few_shot_block(examples):
    lines = []
    for review, label in examples:
        lines.append(f'Ulasan: "{review}"\nSentimen: {label}')
    return "\n\n".join(lines)

def classify_sentiment_llm(review_text):
    few_shot_block = build_few_shot_block(FEW_SHOT_EXAMPLES)
    prompt = f'''Kamu adalah asisten analis sentimen yang ahli menilai ulasan pelanggan e-commerce.
Tugasmu adalah mengklasifikasikan sentimen sebuah ulasan pelanggan ke dalam salah satu dari dua kelas: "positif" atau "negatif".

Berikut beberapa contoh:

{few_shot_block}

Sekarang klasifikasikan ulasan berikut mengikuti pola contoh di atas.
Jawab hanya dengan satu kata, yaitu "positif" atau "negatif", tanpa tambahan kalimat atau tanda baca lain.

Ulasan: "{review_text}"
Sentimen:'''
    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return response.text.strip().lower()

### 5.3 Menjalankan Prediksi pada Test Set

*Output* mentah dari LLM berpotensi mengandung variasi format (misal huruf kapital, spasi berlebih, atau tanda baca tambahan), sehingga perlu dinormalisasi terlebih dahulu agar konsisten dengan label asli (`positif`/`negatif`) sebelum dibandingkan pada tahap evaluasi.

Karena API *key* yang digunakan berada pada *tier* gratis dengan batas jumlah *request* per menit, pemanggilan API ke seluruh data *test* dilakukan dengan jeda antar-*request* serta mekanisme *retry* otomatis apabila terjadi *error* akibat limit tersebut (`429 RESOURCE_EXHAUSTED`).

In [8]:
# helper untuk menangani rate limit & error sementara dari API
# free tier Gemini membatasi jumlah request per menit, sehingga diperlukan jeda dan mekanisme retry.
import time

REQUEST_DELAY_SECONDS = 4.5
MAX_RETRIES = 5
RETRYABLE_ERROR_KEYWORDS = ['429', 'RESOURCE_EXHAUSTED', '503', 'UNAVAILABLE', 'DEADLINE', '500', 'INTERNAL']

def classify_with_retry(review_text, max_retries=MAX_RETRIES):
    for attempt in range(1, max_retries + 1):
        try:
            return classify_sentiment_llm(review_text)
        except Exception as e:
            error_message = str(e)
            if any(keyword in error_message for keyword in RETRYABLE_ERROR_KEYWORDS):
                wait_time = REQUEST_DELAY_SECONDS * attempt
                print(f'Error sementara terdeteksi, mencoba lagi dalam {wait_time:.1f} detik (percobaan {attempt}/{max_retries})...')
                time.sleep(wait_time)
            else:
                raise
    raise RuntimeError(f'Gagal mendapatkan prediksi setelah {max_retries} percobaan untuk ulasan: "{review_text}"')

In [9]:
def normalize_llm_label(raw_label):
    label = raw_label.strip().lower()
    label = label.strip('.').strip()
    if 'positif' in label:
        return 'positif'
    if 'negatif' in label:
        return 'negatif'
    # Fallback apabila model menghasilkan jawaban di luar dua kelas yang diharapkan
    return label

y_pred_llm = []
for i, review in enumerate(X_test):
    pred = classify_with_retry(review)
    y_pred_llm.append(normalize_llm_label(pred))
    time.sleep(REQUEST_DELAY_SECONDS)
    if (i + 1) % 5 == 0:
        print(f'Progres: {i + 1}/{len(X_test)} ulasan telah diproses')

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Progres: 5/40 ulasan telah diproses
Progres: 10/40 ulasan telah diproses
Progres: 15/40 ulasan telah diproses
Progres: 20/40 ulasan telah diproses
Progres: 25/40 ulasan telah diproses
Progres: 30/40 ulasan telah diproses
Progres: 35/40 ulasan telah diproses
Progres: 40/40 ulasan telah diproses


## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [10]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_classic))

=== Model Klasik (Scikit-learn) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
 [[18  0]
 [ 0 22]]


### 6.2 Evaluasi LLM API

In [11]:
print('=== LLM API (Gemini) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_llm))
print(classification_report(y_test, y_pred_llm))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_llm))

=== LLM API (Gemini) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
 [[18  0]
 [ 0 22]]


### 6.3 Tabel Perbandingan Ringkasan

Setelah dilakukan evaluasi dari kedua eksperimen, berikut perbandingan performa antara Model Klasik (Scikit-learn) dan LLM API (Gemini) pada test set yang sama (40 data, 18 negatif dan 22 positif):

| Metrik    | Model Klasik (Scikit-learn) | LLM API (Gemini) |
|-----------|:---------------------------:|:-----------------:|
| Accuracy  | 1.00                         | 1.00               |
| Precision | 1.00                         | 1.00               |
| Recall    | 1.00                         | 1.00               |
| F1-Score  | 1.00                         | 1.00               |

Kedua pendekatan berhasil mengklasifikasikan seluruh 40 data test dengan benar tanpa satu pun kesalahan (lihat confusion matrix pada Section 6.1 dan 6.2), sehingga dari sisi metrik evaluasi keduanya tidak dapat dibedakan pada eksperimen kali ini.

## 7. Analisis Trade-off dan Limitation

**Perbandingan Berdasarkan Performa**

Berdasarkan hasil evaluasi pada Section 6, Model Klasik (Scikit-learn) dan LLM API (Gemini) menunjukkan hasil yang sama. Keduanya memperoleh nilai *accuracy*, *precision*, *recall*, dan F1-Score sebesar 1.00 pada seluruh 40 data *test*, yang terdiri atas 18 ulasan negatif dan 22 ulasan positif. *Confusion matrix* juga menunjukkan bahwa tidak terdapat kesalahan klasifikasi pada kedua pendekatan. Dengan demikian, berdasarkan metrik evaluasi yang digunakan, kedua pendekatan tidak menunjukkan perbedaan performa pada eksperimen ini.

Hasil yang sempurna tersebut kemungkinan dipengaruhi oleh karakteristik *dataset* yang digunakan. Ulasan pelanggan dalam *dataset* cenderung memiliki pola sentimen yang cukup jelas, misalnya melalui penggunaan kata-kata seperti "puas", "kecewa", "lambat", atau "sesuai deskripsi". Selain itu, jumlah data *test* yang digunakan juga relatif kecil, yaitu hanya 40 data. Kondisi tersebut memungkinkan Model Klasik berbasis TF-IDF maupun LLM dengan *few-shot prompting* untuk memisahkan kedua kelas sentimen dengan relatif mudah.

Namun, hasil tersebut belum tentu dapat dipertahankan apabila model diterapkan pada data yang lebih kompleks. Misalnya, model dapat menghadapi kesulitan ketika sebuah ulasan memiliki makna yang ambigu, mengandung pujian sekaligus keluhan dalam satu kalimat, atau menggunakan sarkasme. Oleh karena itu, hasil evaluasi pada eksperimen ini perlu dipahami sesuai dengan karakteristik dan ukuran *dataset* yang digunakan.

**_Trade-off_ Berdasarkan _Effort_ Implementasi, Kecepatan, dan Biaya**

Karena kedua pendekatan menghasilkan performa yang sama pada eksperimen ini, perbandingan keduanya menjadi lebih relevan jika dilihat dari beberapa faktor di luar performa, yaitu *effort* implementasi, kecepatan inferensi, biaya, dan ketergantungan terhadap layanan eksternal.

| Aspek | Model Klasik (Scikit-learn) | LLM API (Gemini) |
|---|---|---|
| *Effort* implementasi | Membutuhkan proses *training*, termasuk *feature extraction* dan *fitting model*, tetapi prosesnya relatif sederhana untuk *task* klasifikasi teks | Tidak membutuhkan proses *training*. Model dapat digunakan dengan merancang *prompt* yang sesuai. Dalam eksperimen ini, digunakan *few-shot prompting* |
| Kecepatan inferensi | Sangat cepat karena proses prediksi dilakukan secara lokal. Seluruh *test set* dapat diproses hampir secara instan | Lebih lambat karena setiap prediksi membutuhkan *request* melalui jaringan ke API. Pada eksperimen ini, pemrosesan 40 data membutuhkan beberapa menit karena terdapat jeda antar-*request* |
| Biaya | Tidak terdapat biaya inferensi setelah model berhasil dilatih | Pada penggunaan produksi terdapat biaya untuk setiap *request*, sedangkan penggunaan *free tier* juga memiliki batasan *rate limit* |
| Ketergantungan pada layanan eksternal | Tidak bergantung pada layanan eksternal karena model dapat berjalan sepenuhnya secara lokal | Bergantung pada ketersediaan dan performa API dari pihak ketiga |

**_Limitation_ yang Ditemukan**

Pada Model Klasik, performa sangat dipengaruhi oleh representasi fitur yang digunakan, dalam eksperimen ini berupa TF-IDF, serta jumlah dan keragaman data *training*. Apabila *dataset* yang digunakan lebih kecil atau memiliki variasi yang lebih tinggi, model dapat mengalami kesulitan dalam mengenali kosakata atau pola baru yang tidak terdapat pada data *training* sebelumnya. Kondisi ini berkaitan dengan kemungkinan munculnya kata-kata yang bersifat *out-of-vocabulary*.

Sementara itu, pendekatan LLM API memiliki beberapa *limitation* teknis yang ditemukan selama eksperimen. Beberapa kendala tersebut meliputi:

- Terjadi *error* `429 RESOURCE_EXHAUSTED` yang disebabkan oleh *rate limit* pada *free tier*. Model yang digunakan memiliki batas maksimal 15 *request* per menit.
- Terjadi *error* `503 UNAVAILABLE` atau *deadline expired* yang bersifat sementara dari sisi *server*. Untuk mengatasinya, diperlukan mekanisme *retry* serta jeda antar-*request* agar proses prediksi dapat diselesaikan.
- *Output* dari LLM berbentuk teks bebas sehingga perlu dilakukan normalisasi terlebih dahulu, seperti penyesuaian huruf kapital, spasi, dan tanda baca, sebelum hasilnya dapat dibandingkan dengan label asli. Hal ini berbeda dengan Model Klasik yang secara langsung menghasilkan label dengan format yang konsisten.

Meskipun terdapat beberapa *limitation* pada kedua pendekatan, tidak ditemukan kesalahan prediksi pada eksperimen ini. *Confusion matrix* dari kedua model menunjukkan hasil yang sempurna, sehingga potensi kesalahan klasifikasi dari masing-masing pendekatan belum dapat dianalisis lebih lanjut berdasarkan eksperimen yang dilakukan.

## 8. Rekomendasi Technical Approach

Berdasarkan hasil eksperimen, kedua pendekatan menghasilkan performa yang sama, dengan *accuracy*, *precision*, *recall*, dan F1-Score masing-masing sebesar 1,00. Oleh karena itu, pemilihan pendekatan tidak didasarkan pada perbedaan performa, tetapi lebih mempertimbangkan *trade-off* yang telah ditemukan pada Section 7, terutama dari sisi *effort* implementasi, kecepatan, biaya, dan *limitation*.

**Model Klasik (Scikit-learn) direkomendasikan sebagai pendekatan utama untuk dikembangkan lebih lanjut menuju sistem produksi.**

Pertimbangan tersebut didasarkan pada beberapa hal berikut:

1. **Kecepatan dan biaya operasional**  
   Fitur klasifikasi sentimen pada halaman produk berpotensi digunakan untuk memproses ulasan dalam jumlah besar dan membutuhkan proses yang relatif cepat. Model klasik yang dapat dijalankan secara lokal memiliki kecepatan inferensi yang tinggi dan tidak membutuhkan biaya untuk setiap *request*. Hal ini membuatnya lebih sesuai untuk penggunaan dalam skala produksi dibandingkan LLM API yang dapat dikenakan biaya per *request* serta memiliki batasan *rate limit*.

2. **Tidak bergantung pada layanan eksternal**  
   Selama eksperimen, pendekatan LLM API mengalami beberapa kendala yang berasal dari faktor eksternal, seperti *rate limit* dan *error* sementara pada *server*. Ketergantungan terhadap layanan eksternal tersebut perlu diperhatikan apabila model akan digunakan dalam sistem produksi yang membutuhkan tingkat *reliability* yang tinggi. Sebaliknya, Model Klasik dapat dijalankan secara lokal sehingga tidak bergantung pada ketersediaan API pihak ketiga.

3. **Performa yang sudah memadai**  
   Pada *dataset* yang digunakan dalam eksperimen ini, Model Klasik mampu mencapai performa yang sama dengan LLM API. Dengan kondisi tersebut, penggunaan pendekatan yang memiliki biaya dan waktu inferensi lebih tinggi belum memberikan peningkatan performa pada hasil eksperimen.

Meskipun demikian, LLM API tetap dapat digunakan pada kondisi tertentu. Salah satu contohnya adalah sebagai *quick win* pada tahap awal pengembangan produk, terutama ketika jumlah data berlabel belum cukup untuk melakukan *training* Model Klasik. LLM juga dapat dipertimbangkan untuk menangani ulasan yang lebih kompleks dan sulit diklasifikasikan menggunakan fitur sederhana seperti TF-IDF, misalnya ulasan yang panjang dan memiliki beberapa konteks atau sentimen yang bercampur.

Dengan demikian, performa Model Klasik tetap perlu dipantau ketika model mulai digunakan pada data produksi yang lebih beragam. Apabila ditemukan penurunan performa pada kondisi nyata, pendekatan *hybrid* dapat menjadi salah satu alternatif. Dalam pendekatan tersebut, Model Klasik tetap menjadi model utama, sedangkan LLM dapat digunakan sebagai *fallback* untuk kasus-kasus yang tingkat keyakinan prediksinya rendah atau sulit diklasifikasikan oleh model utama.